# 02 — Cleaning

Build the reproducible Python ETL pipeline for the US Accidents dataset. Every transformation is logged, and the final cleaned dataset is exported to `data/processed/cleaned_dataset.csv` for downstream EDA and statistical analysis.

**Cleaning is organised around the five problem-statement dimensions:**

1. **Severity** — target variable, must be non-null and typed correctly.
2. **Geography** — state, city, lat/lng preserved for hotspot analysis.
3. **Time** — derive hour, day-of-week, month, season, rush-hour, weekend flags.
4. **Weather** — numeric weather variables imputed with medians (robust to skew).
5. **Infrastructure** — POI booleans (traffic signal, junction, crossing, …) forced to bool.

All step-level logic lives in `scripts/etl_pipeline.py` so it can also be run from the CLI.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.etl_pipeline import (
    load_us_accidents,
    drop_unused_columns,
    parse_datetimes,
    drop_missing_critical,
    impute_weather,
    impute_categorical,
    coerce_booleans,
    add_duration,
    add_time_features,
    add_severity_label,
    handle_outliers,
    save_processed,
    DROP_COLUMNS,
    NUMERIC_WEATHER_COLUMNS,
    CATEGORICAL_FILL_COLUMNS,
    BOOLEAN_POI_COLUMNS,
)

RAW_PATH       = PROJECT_ROOT / 'data' / 'raw' / 'US_Accidents_March23.csv'
PROCESSED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'cleaned_dataset.csv'

# During development you may set SAMPLE_N (e.g. 500_000) to iterate faster.
# Set to None for the final run on the full dataset.
SAMPLE_N: int | None = None

## Step 1 — Load raw data & normalize column names

In [ ]:
df = load_us_accidents(RAW_PATH, sample_n=SAMPLE_N)
rows_initial, cols_initial = df.shape
print(f'Loaded: {rows_initial:,} rows × {cols_initial} columns')
df.head(3)

## Step 2 — Drop columns not used in the government severity analysis

| Column | Reason dropped |
|---|---|
| `id`, `source` | internal identifiers |
| `description` | free text, not structured analysis |
| `end_lat`, `end_lng` | ~50% null, redundant with start coordinates |
| `number` | building number, ~60% null, not analytical |
| `airport_code`, `weather_timestamp` | audit metadata for the weather join |
| `country` | constant = `US` |
| `turning_loop` | constant = `False` in this dataset |
| `nautical_twilight`, `astronomical_twilight` | redundant with `sunrise_sunset` / `civil_twilight` |

In [ ]:
df = drop_unused_columns(df)
print(f'After drop: {df.shape[1]} columns')
print('Columns remaining:', list(df.columns))

## Step 3 — Parse datetimes and drop rows missing critical fields

`severity`, `start_time`, `state`, `start_lat`, `start_lng` are required for every downstream KPI. Rows missing any of these are dropped.

In [ ]:
df = parse_datetimes(df)
before = len(df)
df = drop_missing_critical(df)
print(f'Dropped {before - len(df):,} rows missing critical fields ({(before - len(df)) / before * 100:.2f}%)')

## Step 4 — Impute missing weather values with column medians

Weather variables are right-skewed, so medians are more robust than means.

In [ ]:
missing_before = df[list(NUMERIC_WEATHER_COLUMNS)].isna().sum()
df = impute_weather(df)
missing_after = df[list(NUMERIC_WEATHER_COLUMNS)].isna().sum()
pd.DataFrame({'missing_before': missing_before, 'missing_after': missing_after})

## Step 5 — Fill categorical nulls with `Unknown`

Preserves rows for geographic and temporal aggregations.

In [ ]:
df = impute_categorical(df)
cat_nulls = df[[c for c in CATEGORICAL_FILL_COLUMNS if c in df.columns]].isna().sum()
print('Remaining categorical nulls:')
print(cat_nulls)

## Step 6 — Coerce infrastructure POI flags to boolean

In [ ]:
df = coerce_booleans(df)
print('Boolean POI dtypes:')
print(df[[c for c in BOOLEAN_POI_COLUMNS if c in df.columns]].dtypes)

## Step 7 — Compute `duration_min` and drop impossible durations

Negative durations (end before start) or > 24 h are data-entry errors and are removed.

In [ ]:
before = len(df)
df = add_duration(df)
print(f'Dropped {before - len(df):,} rows with invalid duration')
print('Duration distribution (min):')
print(df['duration_min'].describe().round(2))

## Step 8 — Derive time-based features

`hour`, `day_of_week`, `day_name`, `month`, `month_name`, `year`, `season`, `is_weekend`, `is_rush_hour` — these feed the EDA charts and the Tableau dashboard filters.

In [ ]:
df = add_time_features(df)
df[['start_time', 'hour', 'day_name', 'month_name', 'season', 'is_weekend', 'is_rush_hour']].head()

## Step 9 — Map severity to human-readable labels

| Severity | Label |
|---|---|
| 1 | Low |
| 2 | Moderate |
| 3 | High |
| 4 | Critical |

Also add `is_high_severity` (severity ≥ 3) for hypothesis testing.

In [ ]:
df = add_severity_label(df)
df['severity_label'].value_counts()

## Step 10 — Handle outliers

- Cap `distance_mi` at the 99th percentile (long-tail artefacts).
- Remove temperatures outside the physically possible range (-60 °F to 140 °F).

In [ ]:
before = len(df)
df = handle_outliers(df)
print(f'Dropped {before - len(df):,} rows with impossible temperature values')
print('Distance distribution after capping:')
print(df['distance_mi'].describe().round(3))

## Step 11 — Drop exact duplicates and finalise

In [ ]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f'Dropped {before - len(df):,} exact duplicates')
print(f'Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

## Cleaning audit

Compare row and null counts before and after cleaning.

In [ ]:
rows_final = len(df)
loss_pct = (rows_initial - rows_final) / rows_initial * 100
print(f'Initial rows : {rows_initial:,}')
print(f'Final rows   : {rows_final:,}')
print(f'Row loss     : {rows_initial - rows_final:,} ({loss_pct:.2f}%)')

remaining_nulls = df.isna().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]
print('\nRemaining nulls (should be empty or only in low-priority columns):')
print(remaining_nulls if not remaining_nulls.empty else '  none')

## Export cleaned dataset

In [ ]:
save_processed(df, PROCESSED_PATH)
print(f'✓ Saved cleaned dataset → {PROCESSED_PATH}')
print(f'  Size on disk: {PROCESSED_PATH.stat().st_size / 1e6:.1f} MB')